# Corrective RAG (CRAG) — Tesla Q1 2026 Annual Report

**What you will build:**  
A self-correcting RAG pipeline powered by LangGraph that answers questions about Tesla's Q1 2026 financial results.  
The system goes beyond simple retrieve-and-generate: it **grades** every retrieved document, **falls back to live web search** when the knowledge base comes up short, and **rewrites the question** to improve retrieval quality before retrying.  

**Why Corrective RAG?**  
Plain RAG blindly passes whatever it retrieves into the LLM. If the retrieved chunks are irrelevant, the answer will be wrong or hallucinated.  
CRAG adds three safeguards:
1. A **relevance grader** filters out chunks that do not match the question.
2. A **query rewriter** rephrases the question so retrieval improves on the retry.
3. A **web search fallback** pulls fresh information when the vector store fails entirely.

**Flow at a glance:**
```
User Question
     |
  retrieve  (FAISS over Tesla PDF)
     |
  grade_documents  (LLM scores each chunk: relevant / not relevant)
     |
  +-----------+-------------+
  |                         |
all relevant           any irrelevant
  |                         |
generate             transform_query
                           |
                       web_search
                           |
                        generate
                           |
                         Answer
```

---
**Tech stack (all 2025/2026 stable releases):**
- `langchain >= 1.0`
- `langgraph >= 1.0`
- `langchain-openai` (embeddings + chat)
- `langchain-community` (FAISS, PyPDF, Tavily tool)
- `tavily-python` (web search)
- `pypdf` (PDF text extraction)
- `faiss-cpu` (local vector store)
- `pydantic v2`


## 1. Install dependencies

In [ ]:
# Install all required packages.
# Run this cell once; restart the kernel afterwards before proceeding.
%pip install -q \
    langchain>=1.0 \
    langgraph>=1.0 \
    langchain-openai \
    langchain-community \
    tavily-python \
    pypdf \
    faiss-cpu \
    pydantic>=2.0 \
    python-dotenv \
    requests

## 2. Environment setup

Create a `.env` file in the same directory as this notebook and add the following keys:

```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...
```

- **OpenAI** is used for both embeddings (`text-embedding-3-small`) and the chat LLM (`gpt-4o-mini`).  
- **Tavily** is a search-as-a-service API that returns clean, LLM-ready results. Free tier is available at [tavily.com](https://tavily.com).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current directory

# Validate that both keys are present before going further.
required_keys = ["OPENAI_API_KEY", "TAVILY_API_KEY"]
missing = [k for k in required_keys if not os.getenv(k)]
if missing:
    raise EnvironmentError(
        f"Missing environment variables: {missing}. "
        "Add them to a .env file in this directory."
    )

print("All API keys loaded successfully.")

## 3. Build the vector index from Tesla Q1 2026 PDF

**What happens here:**  
1. Download the Tesla Q1 2026 Investor Update PDF directly from the official IR URL.  
2. Extract text using `PyPDFLoader` — the modern, maintained PDF loader in LangChain.  
3. Chunk the text with `RecursiveCharacterTextSplitter`, which respects natural text boundaries (paragraphs, sentences) before falling back to character splits.  
4. Embed each chunk with `text-embedding-3-small` and store in a local FAISS index.  

`PyPDFLoader` replaces the old `UnstructuredPDFLoader` pattern. It uses `pypdf` under the hood and requires no extra system dependencies.

In [ ]:
import requests
import tempfile
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# --- Download the PDF ---
TESLA_PDF_URL = "https://assets-ir.tesla.com/tesla-contents/IR/TSLA-Q1-2026-Update.pdf"
pdf_path = Path(tempfile.gettempdir()) / "tesla_q1_2026.pdf"

if not pdf_path.exists():
    print("Downloading Tesla Q1 2026 PDF...")
    response = requests.get(TESLA_PDF_URL, timeout=60)
    response.raise_for_status()
    pdf_path.write_bytes(response.content)
    print(f"Saved to {pdf_path}")
else:
    print(f"PDF already on disk: {pdf_path}")

# --- Load pages from PDF ---
# PyPDFLoader returns one Document per page, preserving page-level metadata.
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()
print(f"Loaded {len(pages)} pages from the PDF.")

# --- Chunk into smaller pieces ---
# chunk_size=600: balances context richness vs retrieval precision.
# chunk_overlap=60: avoids cutting sentences mid-thought at boundaries.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=60,
    separators=["\n\n", "\n", " ", ""],
)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks.")

# --- Embed and index ---
# text-embedding-3-small is OpenAI's current recommended small embedding model.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("FAISS index built. Retriever is ready.")

## 4. Retrieval grader

The grader is the core novelty of CRAG. For every chunk retrieved, the LLM answers one binary question: "Is this document relevant to the user's question?"  

We use **structured output** via Pydantic to guarantee the LLM returns a clean `yes` / `no` — no string parsing required.  

`with_structured_output` (available since langchain-openai 0.1+) constrains the LLM to a JSON schema derived from the Pydantic model.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# --- Pydantic schema for structured output ---
class RelevanceScore(BaseModel):
    """Binary relevance decision for a single retrieved chunk."""
    binary_score: str = Field(
        description="'yes' if the document is relevant to the question, 'no' otherwise."
    )

# gpt-4o-mini: fast, cheap, strong enough for binary classification.
grader_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_grader = grader_llm.with_structured_output(RelevanceScore)

grader_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a grader assessing whether a retrieved document chunk is relevant "
        "to a user question about Tesla's financial and operational performance. "
        "If the chunk contains keywords, numbers, or semantic meaning that address "
        "the question, grade it as relevant. Be strict: generic text that merely "
        "mentions Tesla but does not answer the question should be graded 'no'.",
    ),
    (
        "human",
        "Retrieved chunk:\n\n{document}\n\nUser question: {question}",
    ),
])

# Chain: prompt → structured LLM
retrieval_grader = grader_prompt | structured_grader

# --- Quick smoke test ---
test_question = "What was Tesla's total revenue in Q1 2026?"
test_doc = chunks[5].page_content  # pick any chunk for testing
result = retrieval_grader.invoke({"question": test_question, "document": test_doc})
print(f"Grader output: {result.binary_score}")
print(f"Tested chunk preview: {test_doc[:200]}")

## 5. Answer generator

The generator is a standard RAG chain: system prompt + retrieved context + user question → answer.  

Rather than pulling the `rlm/rag-prompt` from LangSmith hub (which requires auth), we define the prompt inline. This makes the notebook fully self-contained and runnable offline (except for API calls).

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# --- Inline RAG prompt ---
# Defined here to avoid any LangSmith hub dependency.
RAG_SYSTEM = (
    "You are an expert financial analyst assistant specializing in Tesla's business. "
    "Use ONLY the provided context to answer the question. "
    "If the context does not contain enough information, say so clearly — do not guess. "
    "Be concise and reference specific numbers or facts from the context where possible."
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

generator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    """Concatenate document chunks into a single context string."""
    return "\n\n---\n\n".join(
        f"[Page {d.metadata.get('page', '?')}]\n{d.page_content}" for d in docs
    )

# LCEL chain: prompt | LLM | string parser
rag_chain = rag_prompt | generator_llm | StrOutputParser()

print("Generator chain ready.")

## 6. Query rewriter

When the grader finds that retrieved chunks are irrelevant, the pipeline does not give up — it rewrites the question to be more precise before triggering a web search.  

**Why rewrite before web search?**  
The original question may use domain shorthand (e.g., "Q1 earnings") that a web search engine handles poorly. The rewriter expands it to a version optimised for retrieval (e.g., "Tesla Q1 2026 total revenue and operating income").

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

rewriter_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a query rewriter. Your job is to take an imprecise or ambiguous question "
        "and produce a cleaner version that is better suited for web search. "
        "Focus on surfacing the core intent and adding relevant keywords. "
        "Output only the improved question — no explanation, no preamble.",
    ),
    (
        "human",
        "Original question: {question}\n\nImproved question:",
    ),
])

rewriter_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
question_rewriter = rewriter_prompt | rewriter_llm | StrOutputParser()

# Smoke test
rewritten = question_rewriter.invoke({"question": "How did Tesla do last quarter?"})
print(f"Rewritten: {rewritten}")

## 7. Web search tool

`TavilySearchResults` is a managed web search tool. It calls Tavily's API and returns LLM-ready results (no HTML scraping, no JavaScript execution, clean text).  

`k=3` returns three results, which is sufficient context without overwhelming the LLM's context window.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# k=3: three web results per query is a good balance of coverage vs. token cost.
web_search_tool = TavilySearchResults(k=3)

print("Web search tool ready.")

## 8. Graph state definition

LangGraph is a stateful graph framework. Every node in the graph reads from and writes to a shared **state dictionary**.  

We define the state as a `TypedDict` with four fields:

| Field | Type | Purpose |
|---|---|---|
| `question` | `str` | The current (possibly rewritten) question |
| `generation` | `str` | The LLM's final answer |
| `web_search_needed` | `str` | `"Yes"` or `"No"` flag set by the grader |
| `documents` | `List[Document]` | Chunks currently in context |

In [ ]:
from typing import List
from typing_extensions import TypedDict
from langchain_core.documents import Document

class CRAGState(TypedDict):
    """
    Shared state passed between all nodes in the CRAG graph.

    question         : The user's question (may be rewritten mid-graph).
    generation       : The final answer produced by the generator node.
    web_search_needed: 'Yes' if any retrieved chunk was graded irrelevant;
                       'No' if all chunks passed the relevance check.
    documents        : List of Document objects currently in context.
    """
    question: str
    generation: str
    web_search_needed: str
    documents: List[Document]

print("State schema defined.")

## 9. Graph nodes

Each node is a plain Python function that accepts a `CRAGState` dict and returns a partial update to that dict.  
LangGraph merges the returned dict into the current state — you only need to return the fields you change.

In [ ]:
# ─────────────────────────────────────────────
# NODE 1 — retrieve
# ─────────────────────────────────────────────
def retrieve(state: CRAGState) -> dict:
    """
    Retrieve the top-k most similar chunks from the FAISS index.

    Input state fields used : question
    Output state fields set : documents
    """
    print("\n--- NODE: retrieve ---")
    question = state["question"]
    documents = retriever.invoke(question)
    print(f"Retrieved {len(documents)} chunks.")
    return {"documents": documents, "question": question}


# ─────────────────────────────────────────────
# NODE 2 — grade_documents
# ─────────────────────────────────────────────
def grade_documents(state: CRAGState) -> dict:
    """
    Score each retrieved chunk for relevance.

    - Chunks graded 'yes' are kept in the documents list.
    - If any chunk is graded 'no', the web_search_needed flag is set to 'Yes',
      signalling the conditional edge to route toward transform_query.

    Input state fields used : question, documents
    Output state fields set : documents (filtered), web_search_needed
    """
    print("\n--- NODE: grade_documents ---")
    question = state["question"]
    documents = state["documents"]

    relevant_docs = []
    web_search_needed = "No"

    for doc in documents:
        score = retrieval_grader.invoke(
            {"question": question, "document": doc.page_content}
        )
        if score.binary_score == "yes":
            print("  RELEVANT chunk kept.")
            relevant_docs.append(doc)
        else:
            print("  IRRELEVANT chunk dropped — will trigger web search.")
            web_search_needed = "Yes"

    print(f"Relevant chunks after grading: {len(relevant_docs)} / {len(documents)}")
    return {
        "documents": relevant_docs,
        "question": question,
        "web_search_needed": web_search_needed,
    }


# ─────────────────────────────────────────────
# NODE 3 — transform_query
# ─────────────────────────────────────────────
def transform_query(state: CRAGState) -> dict:
    """
    Rewrite the user question to be more precise for web search.

    Input state fields used : question, documents
    Output state fields set : question (updated)
    """
    print("\n--- NODE: transform_query ---")
    question = state["question"]
    better_question = question_rewriter.invoke({"question": question})
    print(f"Original : {question}")
    print(f"Rewritten: {better_question}")
    return {"question": better_question, "documents": state["documents"]}


# ─────────────────────────────────────────────
# NODE 4 — web_search
# ─────────────────────────────────────────────
def web_search(state: CRAGState) -> dict:
    """
    Supplement the context with live web search results.

    Tavily results are converted into a single Document and appended
    to whatever relevant chunks survived the grading step.

    Input state fields used : question, documents
    Output state fields set : documents (extended)
    """
    print("\n--- NODE: web_search ---")
    question = state["question"]
    documents = state["documents"]

    search_results = web_search_tool.invoke({"query": question})

    # Each result is a dict with 'content' and 'url' keys.
    combined_text = "\n\n".join(
        f"[Source: {r.get('url', 'unknown')}]\n{r['content']}"
        for r in search_results
    )
    web_doc = Document(
        page_content=combined_text,
        metadata={"source": "tavily_web_search"},
    )
    documents = documents + [web_doc]  # keep existing relevant chunks + web results
    print(f"Web search added {len(search_results)} results.")
    return {"documents": documents, "question": question}


# ─────────────────────────────────────────────
# NODE 5 — generate
# ─────────────────────────────────────────────
def generate(state: CRAGState) -> dict:
    """
    Generate a final answer from the (graded and possibly web-augmented) context.

    Input state fields used : question, documents
    Output state fields set : generation
    """
    print("\n--- NODE: generate ---")
    question = state["question"]
    documents = state["documents"]

    generation = rag_chain.invoke({
        "context": format_docs(documents),
        "question": question,
    })
    print("Answer generated.")
    return {"generation": generation, "documents": documents, "question": question}


print("All 5 nodes defined.")

## 10. Conditional edge — routing logic

After `grade_documents`, the graph needs to decide which path to take.  
This is expressed as a **conditional edge**: a function that returns a string key, which LangGraph maps to the next node.

In [ ]:
def decide_next_step(state: CRAGState) -> str:
    """
    Routing function called after grade_documents.

    Returns
    -------
    'transform_query' : if any chunk was irrelevant (web search needed)
    'generate'        : if all chunks are relevant (proceed directly to answer)
    """
    print("\n--- EDGE: decide_next_step ---")
    if state["web_search_needed"] == "Yes":
        print("Decision: some chunks were irrelevant → rewrite query and search the web.")
        return "transform_query"
    else:
        print("Decision: all chunks are relevant → generate answer directly.")
        return "generate"

print("Conditional edge function defined.")

## 11. Build and compile the LangGraph workflow

The graph is assembled in three steps:
1. **Add nodes** — register each function under a name.
2. **Add edges** — define the fixed paths between nodes.
3. **Add conditional edges** — register the routing function and its possible destinations.
4. **Compile** — LangGraph validates the graph topology and returns a runnable.

In [ ]:
from langgraph.graph import END, StateGraph, START

# --- Instantiate the graph with our state schema ---
graph_builder = StateGraph(CRAGState)

# --- Register nodes ---
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("grade_documents", grade_documents)
graph_builder.add_node("generate", generate)
graph_builder.add_node("transform_query", transform_query)
graph_builder.add_node("web_search_node", web_search)

# --- Fixed edges ---
# The graph always starts at retrieve.
graph_builder.add_edge(START, "retrieve")

# retrieve → grade_documents is always the next step.
graph_builder.add_edge("retrieve", "grade_documents")

# After rewriting the query, always go to web search.
graph_builder.add_edge("transform_query", "web_search_node")

# After web search, always generate the answer.
graph_builder.add_edge("web_search_node", "generate")

# After generating, the conversation ends.
graph_builder.add_edge("generate", END)

# --- Conditional edge after grade_documents ---
# The routing function decide_next_step returns 'transform_query' or 'generate'.
graph_builder.add_conditional_edges(
    "grade_documents",          # source node
    decide_next_step,           # routing function
    {
        "transform_query": "transform_query",
        "generate": "generate",
    },
)

# --- Compile ---
crag_app = graph_builder.compile()

print("Graph compiled successfully.")

## 12. Visualise the graph

LangGraph can render the workflow as a Mermaid diagram, which gives students a clear picture of all paths through the system.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(crag_app.get_graph(xray=True).draw_mermaid_png()))
except Exception as e:
    # draw_mermaid_png requires the 'playwright' or 'pyppeteer' package.
    # Fall back to printing the Mermaid source instead.
    print("Could not render PNG (install playwright for visual output).")
    print("Mermaid diagram source:")
    print(crag_app.get_graph().draw_mermaid())

## 13. Run the pipeline — Test questions

We test three question categories:

| Category | Expected path |
|---|---|
| Directly in the PDF (financial data) | retrieve → grade (all relevant) → generate |
| Partially in the PDF | retrieve → grade (some irrelevant) → rewrite → web search → generate |
| Not in the PDF at all | retrieve → grade (none relevant) → rewrite → web search → generate |

In [ ]:
def run_crag(question: str) -> str:
    """
    Run the CRAG pipeline and return the final answer.

    Parameters
    ----------
    question : str  The user's question.

    Returns
    -------
    str  The generated answer.
    """
    print("=" * 60)
    print(f"QUESTION: {question}")
    print("=" * 60)

    initial_state: CRAGState = {
        "question": question,
        "generation": "",
        "web_search_needed": "No",
        "documents": [],
    }

    final_state = crag_app.invoke(initial_state)

    print("\n" + "=" * 60)
    print("ANSWER:")
    print(final_state["generation"])
    print("=" * 60 + "\n")

    return final_state["generation"]

In [ ]:
# Test 1 — Question directly answered by the Tesla PDF
# Expected path: retrieve → grade (relevant) → generate
answer1 = run_crag("What was Tesla's total revenue in Q1 2026?")

In [ ]:
# Test 2 — Question answered by the PDF but requiring some inference
# Expected path: retrieve → grade → (possibly) web search → generate
answer2 = run_crag("How does Tesla's Q1 2026 free cash flow compare to Q1 2025?")

In [ ]:
# Test 3 — Question not well covered by the PDF (triggers web search)
# Expected path: retrieve → grade (some/all irrelevant) → rewrite → web search → generate
answer3 = run_crag("What did Wall Street analysts say about Tesla's Q1 2026 earnings?")

## 14. Streaming execution (optional — advanced)

LangGraph supports streaming, which lets you observe every state transition as it happens. This is useful for debugging and for building interactive UIs where you want to show the user what the agent is doing in real time.

In [ ]:
question = "What are Tesla's plans for the Optimus robot in 2026?"

print(f"Streaming CRAG for: '{question}'\n")

initial_state: CRAGState = {
    "question": question,
    "generation": "",
    "web_search_needed": "No",
    "documents": [],
}

# stream() yields the state dict after each node completes.
for step_output in crag_app.stream(initial_state):
    # step_output is a dict: { node_name: updated_state_fields }
    node_name = list(step_output.keys())[0]
    print(f"[completed node: {node_name}]")

    if node_name == "generate":
        print("\nFinal Answer:")
        print(step_output[node_name]["generation"])

## 15. Recap — What each component did

| Component | Role | Key detail |
|---|---|---|
| `PyPDFLoader` | Load Tesla PDF pages | Modern replacement for UnstructuredPDFLoader |
| `RecursiveCharacterTextSplitter` | Chunk text | Respects natural boundaries before splitting by characters |
| `OpenAIEmbeddings(text-embedding-3-small)` | Vectorise chunks | Faster and cheaper than ada-002 |
| `FAISS` | Local vector store | In-memory, no external DB needed |
| `RelevanceScore` (Pydantic) | Structured grader output | Eliminates string parsing, guarantees yes/no |
| `with_structured_output` | Force LLM to return schema | Pydantic v2 compatible |
| `TavilySearchResults` | Web fallback | Returns clean text, no scraping |
| `question_rewriter` | Improve retrieval | Better queries → better web results |
| `StateGraph` | Orchestrate the flow | LangGraph v1.0 API |
| `add_conditional_edges` | Route after grading | The core CRAG decision point |

---

**The key insight of Corrective RAG:**  
Retrieval is not a black box you trust blindly. By adding a grading step, the system becomes self-aware of its own knowledge gaps and has a defined protocol to fill them — either by improving the question or reaching for external sources.